# 03 — Prétraitement des données

## Projet : Smart City Energy Forecasting — Tetouan

## 1. Importation des bibliothèques et des fonctions `src`

In [1]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import (
    load_tetouan_data,
    audit_tetouan_data,
    dataset_quality_summary,
)

from src.preprocessing import (
    FORBIDDEN_DIRECT_FEATURES,
    check_10min_measurements_per_hour,
    resample_hourly,
    find_missing_hours,
    handle_missing_values,
    check_physical_ranges,
    summarize_outliers,
    annotate_load_outliers,
    validate_clean_hourly_dataset,
    temporal_train_val_test_split,
    get_base_feature_columns,
    scale_train_val_test,
    save_preprocessing_outputs,
    save_scalers,
)

## 2. Définition des chemins du projet

In [2]:
DATA_FILENAME = "Tetuan City power consumption.csv"

RAW_PATH = PROJECT_ROOT / "data" / "raw" / DATA_FILENAME
if not RAW_PATH.exists():
    RAW_PATH = PROJECT_ROOT / DATA_FILENAME

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results" / "preprocessing"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Fichier brut introuvable : {RAW_PATH}")

print(f"Racine du projet détectée : {PROJECT_ROOT}")
print(f"Fichier brut trouvé       : {RAW_PATH}")
print(f"Dossier processed         : {PROCESSED_DIR}")
print(f"Dossier models            : {MODELS_DIR}")
print(f"Dossier results           : {RESULTS_DIR}")

Racine du projet détectée : C:\Users\Badr\UB\0_Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan
Fichier brut trouvé       : C:\Users\Badr\UB\0_Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\data\raw\Tetuan City power consumption.csv
Dossier processed         : C:\Users\Badr\UB\0_Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\data\processed
Dossier models            : C:\Users\Badr\UB\0_Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\models
Dossier results           : C:\Users\Badr\UB\0_Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\results\preprocessing


## 3. Chargement standardisé du dataset Tetouan

In [3]:
raw_df_info = pd.read_csv(RAW_PATH)
raw_shape = raw_df_info.shape
raw_columns = raw_df_info.columns.tolist()

print(f"Dimensions brutes du CSV : {raw_shape}")
print("Colonnes originales :")
print(raw_columns)

df = load_tetouan_data(
    RAW_PATH,
    set_datetime_index=True,
    strict_frequency=True,
)

print("\nRésumé qualité global :")
display(pd.Series(dataset_quality_summary(df), name="value").to_frame())

print("\nAudit qualité par variable :")
display(audit_tetouan_data(df))

print("\nAperçu du dataset chargé :")
display(df.head())

Dimensions brutes du CSV : (52416, 9)
Colonnes originales :
['DateTime', 'Temperature', 'Humidity', 'Wind Speed', 'general diffuse flows', 'diffuse flows', 'Zone 1 Power Consumption', 'Zone 2  Power Consumption', 'Zone 3  Power Consumption']

Résumé qualité global :


,value
n_rows,52416
n_columns,10
start_date,2017-01-01 00:00:00
end_date,2017-12-30 23:50:00
inferred_frequency,10min
missing_total,0
duplicated_datetime,0



Audit qualité par variable :


,dtype,missing_count,missing_rate_pct,min,mean,50%,max,std
temperature,float64,0,0.0,3.247000,18.810024,18.780000,40.01000,5.815476
humidity,float64,0,0.0,11.340000,68.259518,69.860000,94.80000,15.551177
wind_speed,float64,0,0.0,0.050000,1.959489,0.086000,6.48300,2.348862
general_diffuse_flows,float64,0,0.0,0.004000,182.696614,5.035500,1163.00000,264.400960
diffuse_flows,float64,0,0.0,0.011000,75.028022,4.456000,936.00000,124.210949
zone1_power,float64,0,0.0,13895.696200,32344.970564,32265.920340,52204.39512,7130.562564
zone2_power,float64,0,0.0,8560.081466,21042.509082,20823.168405,37408.86076,5201.465892
zone3_power,float64,0,0.0,5935.174070,17835.406218,16415.117470,47598.32636,6622.165099
target,float64,0,0.0,13895.696200,32344.970564,32265.920340,52204.39512,7130.562564
total_load,float64,0,0.0,36785.039739,71222.885864,69788.790940,134208.14595,17143.138964



Aperçu du dataset chargé :


,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows,zone1_power,zone2_power,zone3_power,target,total_load
datetime,,,,,,,,,,
2017-01-01 00:00:00,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,34055.69620,70425.53544
2017-01-01 00:10:00,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,29814.68354,69320.84387
2017-01-01 00:20:00,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,29128.10127,67803.22193
2017-01-01 00:30:00,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,28228.86076,65489.23209
2017-01-01 00:40:00,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,27335.69620,63650.44627


## 4. Contrôle de cohérence de `target` et `total_load`

In [4]:
target_check = np.allclose(df["target"], df["zone1_power"])
total_load_check = np.allclose(
    df["total_load"],
    df["zone1_power"] + df["zone2_power"] + df["zone3_power"],
)

print(f"Vérification target = zone1_power : {target_check}")
print(f"Vérification total_load = somme des trois zones : {total_load_check}")

if not target_check:
    raise ValueError("La variable target n'est pas égale à zone1_power.")

if not total_load_check:
    raise ValueError("La variable total_load est incorrecte.")

Vérification target = zone1_power : True
Vérification total_load = somme des trois zones : True


## 5. Rééchantillonnage horaire et contrôle temporel

In [5]:
hourly_counts = check_10min_measurements_per_hour(
    df,
    expected_per_hour=6,
    raise_error=True,
)

hourly_counts_distribution = hourly_counts.value_counts().sort_index()
incomplete_hours = hourly_counts[hourly_counts != 6]

df_hourly = resample_hourly(
    df,
    check_counts=False,
)

missing_hours = find_missing_hours(df_hourly)

print("Distribution du nombre de mesures 10 minutes par heure :")
display(hourly_counts_distribution.to_frame(name="nombre_d_heures"))

print(f"Dimensions après rééchantillonnage horaire : {df_hourly.shape}")
print(f"Fréquence horaire inférée                 : {pd.infer_freq(df_hourly.index)}")
print(f"Valeurs manquantes après resampling       : {df_hourly.isna().sum().sum()}")
print(f"Heures manquantes                         : {len(missing_hours)}")
print(f"Heures avec nombre de mesures incorrect   : {len(incomplete_hours)}")

display(df_hourly.head())

Distribution du nombre de mesures 10 minutes par heure :


,nombre_d_heures
6,8736


Dimensions après rééchantillonnage horaire : (8736, 10)
Fréquence horaire inférée                 : h
Valeurs manquantes après resampling       : 0
Heures manquantes                         : 0
Heures avec nombre de mesures incorrect   : 0


,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows,zone1_power,zone2_power,zone3_power,target,total_load
datetime,,,,,,,,,,
2017-01-01 00:00:00,6.196833,75.066667,0.081833,0.063500,0.098833,29197.974683,18026.747720,19252.048193,29197.974683,66476.770597
2017-01-01 01:00:00,5.548833,77.583333,0.082000,0.056833,0.112500,24657.215190,16078.419453,17042.891567,24657.215190,57778.526210
2017-01-01 02:00:00,5.054333,78.933333,0.082333,0.063000,0.129167,22083.037973,14330.699088,15676.144578,22083.037973,52089.881640
2017-01-01 03:00:00,5.004333,77.083333,0.082833,0.059833,0.141000,20811.139240,13219.452887,14883.855422,20811.139240,48914.447548
2017-01-01 04:00:00,5.097667,74.050000,0.082333,0.058000,0.122833,20475.949367,12921.580547,14317.108433,20475.949367,47714.638347


## 6. Gestion sécurisée des valeurs manquantes

In [6]:
missing_before = int(df_hourly.isna().sum().sum())

df_clean = handle_missing_values(
    df_hourly,
    interpolation_limit=3,
    ffill_limit=3,
    drop_remaining=True,
)

missing_after = int(df_clean.isna().sum().sum())

print(f"Valeurs manquantes avant traitement : {missing_before}")
print(f"Valeurs manquantes après traitement : {missing_after}")
print(f"Dimensions après gestion des valeurs manquantes : {df_clean.shape}")

Valeurs manquantes avant traitement : 0
Valeurs manquantes après traitement : 0
Dimensions après gestion des valeurs manquantes : (8736, 10)


## 7. Contrôle des plages physiques et détection exploratoire des anomalies

In [7]:
range_checks = check_physical_ranges(df_clean)

cols_to_check = [
    "temperature",
    "humidity",
    "wind_speed",
    "general_diffuse_flows",
    "diffuse_flows",
    "zone1_power",
    "zone2_power",
    "zone3_power",
    "target",
    "total_load",
]

outlier_summary = summarize_outliers(
    df_clean,
    columns=cols_to_check,
    window=24,
    threshold=3.5,
)

print("Contrôle des plages physiques :")
display(range_checks.to_frame())

print("\nRésumé des anomalies par Rolling Z-Score :")
display(outlier_summary)

Contrôle des plages physiques :


,count
negative_zone1_power,0
negative_zone2_power,0
negative_zone3_power,0
negative_target,0
negative_total_load,0
humidity_out_of_range,0
temperature_out_of_range,0
negative_wind_speed,0



Résumé des anomalies par Rolling Z-Score :


,outlier_count
wind_speed,259
diffuse_flows,84
humidity,67
general_diffuse_flows,40
temperature,38
zone3_power,1
zone1_power,0
zone2_power,0
target,0
total_load,0


## 8. Annotation des pics de charge et validation finale stricte

In [8]:
df_clean = annotate_load_outliers(
    df_clean,
    target_col="target",
    total_load_col="total_load",
    window=24,
    threshold=3.5,
    fill_initial_zscore=0.0,
)

validation_report = validate_clean_hourly_dataset(df_clean)

print(f"Pics/anomalies sur target détectés       : {df_clean['is_target_outlier'].sum()}")
print(f"Pics/anomalies sur total_load détectés   : {df_clean['is_total_load_outlier'].sum()}")
print(f"Pics/anomalies de charge globalement     : {df_clean['is_load_outlier'].sum()}")

print("\nValidation finale stricte :")
display(pd.Series(validation_report, name="value").to_frame())

print("\nAperçu du dataset propre :")
display(df_clean.head())
display(df_clean.tail())

Pics/anomalies sur target détectés       : 0
Pics/anomalies sur total_load détectés   : 0
Pics/anomalies de charge globalement     : 0

Validation finale stricte :


,value
missing_total,0
duplicated_total,0
inferred_frequency,h
n_rows,8736
n_columns,15
start_date,2017-01-01 00:00:00
end_date,2017-12-30 23:00:00



Aperçu du dataset propre :


,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows,zone1_power,zone2_power,zone3_power,target,total_load,target_rolling_zscore,total_load_rolling_zscore,is_target_outlier,is_total_load_outlier,is_load_outlier
datetime,,,,,,,,,,,,,,,
2017-01-01 00:00:00,6.196833,75.066667,0.081833,0.063500,0.098833,29197.974683,18026.747720,19252.048193,29197.974683,66476.770597,0.0,0.0,0,0,0
2017-01-01 01:00:00,5.548833,77.583333,0.082000,0.056833,0.112500,24657.215190,16078.419453,17042.891567,24657.215190,57778.526210,0.0,0.0,0,0,0
2017-01-01 02:00:00,5.054333,78.933333,0.082333,0.063000,0.129167,22083.037973,14330.699088,15676.144578,22083.037973,52089.881640,0.0,0.0,0,0,0
2017-01-01 03:00:00,5.004333,77.083333,0.082833,0.059833,0.141000,20811.139240,13219.452887,14883.855422,20811.139240,48914.447548,0.0,0.0,0,0,0
2017-01-01 04:00:00,5.097667,74.050000,0.082333,0.058000,0.122833,20475.949367,12921.580547,14317.108433,20475.949367,47714.638347,0.0,0.0,0,0,0


,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows,zone1_power,zone2_power,zone3_power,target,total_load,target_rolling_zscore,total_load_rolling_zscore,is_target_outlier,is_total_load_outlier,is_load_outlier
datetime,,,,,,,,,,,,,,,
2017-12-30 19:00:00,9.453333,62.406667,0.074667,0.052000,0.102000,37513.814957,32497.698680,16926.770708,37513.814957,86938.284345,1.517436,1.611855,0,0,0
2017-12-30 20:00:00,9.041667,63.990000,0.080333,0.052667,0.105000,37008.871988,32020.251610,16998.799520,37008.871988,86027.923118,1.454852,1.558531,0,0,0
2017-12-30 21:00:00,8.011667,69.675000,0.081500,0.073167,0.098333,35358.174905,30757.901197,16608.883553,35358.174905,82724.959655,1.204273,1.339369,0,0,0
2017-12-30 22:00:00,7.598333,70.315000,0.081833,0.058667,0.108167,33993.409380,28477.447070,15614.885955,33993.409380,78085.742405,0.995139,1.021828,0,0,0
2017-12-30 23:00:00,6.877500,72.900000,0.081500,0.060333,0.092667,30107.984788,25713.409022,14143.577428,30107.984788,69964.971238,0.331566,0.434969,0,0,0


## 9. Split chronologique train / validation / test

In [9]:
train_df, val_df, test_df = temporal_train_val_test_split(
    df_clean,
    train_size=0.70,
    val_size=0.15,
)

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "start_date": [train_df.index.min(), val_df.index.min(), test_df.index.min()],
        "end_date": [train_df.index.max(), val_df.index.max(), test_df.index.max()],
        "n_rows": [len(train_df), len(val_df), len(test_df)],
        "n_columns": [train_df.shape[1], val_df.shape[1], test_df.shape[1]],
    }
)

display(split_summary)

,split,start_date,end_date,n_rows,n_columns
0,train,2017-01-01 00:00:00,2017-09-12 18:00:00,6115,15
1,validation,2017-09-12 19:00:00,2017-11-06 08:00:00,1310,15
2,test,2017-11-06 09:00:00,2017-12-30 23:00:00,1311,15


## 10. Normalisation de base sans data leakage

In [10]:
TARGET_COL = "target"

feature_cols = get_base_feature_columns(df_clean)

scaled_outputs = scale_train_val_test(
    train_df,
    val_df,
    test_df,
    feature_cols=feature_cols,
    target_col=TARGET_COL,
    scaler_type="standard",
)

X_train_scaled_df = scaled_outputs["X_train"]
X_val_scaled_df = scaled_outputs["X_val"]
X_test_scaled_df = scaled_outputs["X_test"]

y_train_scaled_df = scaled_outputs["y_train"]
y_val_scaled_df = scaled_outputs["y_val"]
y_test_scaled_df = scaled_outputs["y_test"]

scaler_X = scaled_outputs["scaler_X"]
scaler_y = scaled_outputs["scaler_y"]

print(f"Variables explicatives utilisées : {feature_cols}")
print(f"X_train_scaled shape : {X_train_scaled_df.shape}")
print(f"X_val_scaled shape   : {X_val_scaled_df.shape}")
print(f"X_test_scaled shape  : {X_test_scaled_df.shape}")
print(f"y_train_scaled shape : {y_train_scaled_df.shape}")
print(f"y_val_scaled shape   : {y_val_scaled_df.shape}")
print(f"y_test_scaled shape  : {y_test_scaled_df.shape}")

print("\nAperçu X_train normalisé :")
display(X_train_scaled_df.head())

print("\nAperçu y_train normalisé :")
display(y_train_scaled_df.head())

Variables explicatives utilisées : ['temperature', 'humidity', 'wind_speed', 'general_diffuse_flows', 'diffuse_flows']
X_train_scaled shape : (6115, 5)
X_val_scaled shape   : (1310, 5)
X_test_scaled shape  : (1311, 5)
y_train_scaled shape : (6115, 1)
y_val_scaled shape   : (1310, 1)
y_test_scaled shape  : (1311, 1)

Aperçu X_train normalisé :


,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows
datetime,,,,,
2017-01-01 00:00:00,-2.112482,0.462947,-0.815577,-0.735586,-0.661449
2017-01-01 01:00:00,-2.217439,0.620177,-0.815506,-0.735609,-0.661344
2017-01-01 02:00:00,-2.297534,0.704519,-0.815364,-0.735588,-0.661217
2017-01-01 03:00:00,-2.305632,0.588939,-0.815151,-0.735599,-0.661126
2017-01-01 04:00:00,-2.290515,0.399430,-0.815364,-0.735605,-0.661265



Aperçu y_train normalisé :


,target
datetime,
2017-01-01 00:00:00,-0.522653
2017-01-01 01:00:00,-1.154081
2017-01-01 02:00:00,-1.512040
2017-01-01 03:00:00,-1.688908
2017-01-01 04:00:00,-1.735518


## 11. Sauvegarde des datasets, fichiers normalisés, scalers et rapport JSON

In [11]:
saved_data_paths = save_preprocessing_outputs(
    output_dir=PROCESSED_DIR,
    clean_df=df_clean,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    scaled_outputs=scaled_outputs,
)

saved_scaler_paths = save_scalers(
    scaler_X=scaler_X,
    scaler_y=scaler_y,
    models_dir=MODELS_DIR,
    prefix="base",
)

feature_engineering_rules = {
    "allowed_consumption_usage": [
        "target_lag_1h",
        "target_lag_24h",
        "target_lag_168h",
        "rolling_features_with_shift_1",
    ],
    "allowed_other_features": [
        "weather_features",
        "calendar_features",
        "cyclical_time_encoding",
        "holiday_features_if_available",
    ],
    "forbidden_direct_features": FORBIDDEN_DIRECT_FEATURES,
    "main_rule": (
        "Les colonnes de consommation instantanée sont conservées pour construire "
        "des lags et rolling features, mais ne doivent pas être utilisées directement "
        "comme variables explicatives au même timestamp."
    ),
}

preprocessing_report = {
    "raw_shape": list(raw_shape),
    "loaded_shape_with_target_total_load": list(df.shape),
    "hourly_shape": list(df_hourly.shape),
    "clean_hourly_shape": list(df_clean.shape),
    "train_shape": list(train_df.shape),
    "val_shape": list(val_df.shape),
    "test_shape": list(test_df.shape),
    "raw_start_date": str(df.index.min()),
    "raw_end_date": str(df.index.max()),
    "hourly_start_date": str(df_clean.index.min()),
    "hourly_end_date": str(df_clean.index.max()),
    "raw_inferred_frequency": str(pd.infer_freq(df.index)),
    "hourly_inferred_frequency": str(pd.infer_freq(df_clean.index)),
    "invalid_dates": int(df.attrs.get("invalid_dates", 0)),
    "duplicates_before": int(df.attrs.get("duplicated_datetime_count", 0)),
    "duplicates_final": int(df_clean.index.duplicated().sum()),
    "missing_before_handling": int(missing_before),
    "missing_after_cleaning": int(df_clean.isna().sum().sum()),
    "missing_hours": int(len(missing_hours)),
    "hourly_counts_distribution": {
        str(k): int(v) for k, v in hourly_counts_distribution.items()
    },
    "incomplete_hours_after_10min_check": int(len(incomplete_hours)),
    "expected_measurements_per_hour": 6,
    "range_checks": {str(k): int(v) for k, v in range_checks.items()},
    "outlier_summary": {
        str(k): int(v) for k, v in outlier_summary["outlier_count"].items()
    },
    "target_outliers": int(df_clean["is_target_outlier"].sum()),
    "total_load_outliers": int(df_clean["is_total_load_outlier"].sum()),
    "load_outliers": int(df_clean["is_load_outlier"].sum()),
    "target_column": TARGET_COL,
    "feature_columns_base": feature_cols,
    "forbidden_direct_features": FORBIDDEN_DIRECT_FEATURES,
    "feature_engineering_rules": feature_engineering_rules,
    "strict_validation": validation_report,
    "saved_files": {
        **saved_data_paths,
        **saved_scaler_paths,
    },
}

report_path = RESULTS_DIR / "preprocessing_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(preprocessing_report, f, indent=4, ensure_ascii=False)

print("Fichiers sauvegardés :")
for name, path in saved_data_paths.items():
    print(f"- {name}: {path}")
for name, path in saved_scaler_paths.items():
    print(f"- {name}: {path}")
print(f"- preprocessing_report: {report_path}")

Fichiers sauvegardés :
- clean_full: C:\Users\Badr\UB\0_Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\data\processed\tetouan_hourly_clean.csv
- train_clean: C:\Users\Badr\UB\0_Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\data\processed\train_clean.csv
- val_clean: C:\Users\Badr\UB\0_Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\data\processed\val_clean.csv
- test_clean: C:\Users\Badr\UB\0_Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\data\processed\test_clean.csv
- X_train: C:\Users\Badr\UB\0_Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\data\processed\X_train_scaled_base.csv
- X_val: C:\Users\Badr\UB\0_Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\data\processed\X_val_scaled_base.csv
- X_test: C:\Users\Badr\UB\0_Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\data\processed\X_tes

## 12. Résumé final du preprocessing

In [12]:
print("=" * 80)
print("RÉSUMÉ FINAL DU PREPROCESSING")
print("=" * 80)

print(f"Dimensions brutes CSV                  : {raw_shape}")
print(f"Dimensions chargées avec target/total   : {df.shape}")
print(f"Dimensions après resampling horaire     : {df_hourly.shape}")
print(f"Dimensions finales propres              : {df_clean.shape}")

print("-" * 80)
print(f"Période brute                           : {df.index.min()} → {df.index.max()}")
print(f"Période horaire                         : {df_clean.index.min()} → {df_clean.index.max()}")
print(f"Fréquence brute                         : {pd.infer_freq(df.index)}")
print(f"Fréquence finale                        : {pd.infer_freq(df_clean.index)}")

print("-" * 80)
print(f"Valeurs manquantes avant traitement     : {missing_before}")
print(f"Valeurs manquantes finales              : {df_clean.isna().sum().sum()}")
print(f"Heures manquantes                       : {len(missing_hours)}")
print(f"Heures avec nb mesures 10min incorrect : {len(incomplete_hours)}")
print(f"Doublons temporels initiaux             : {int(df.attrs.get('duplicated_datetime_count', 0))}")
print(f"Doublons temporels finaux               : {df_clean.index.duplicated().sum()}")

print("-" * 80)
print(f"Anomalies target conservées             : {df_clean['is_target_outlier'].sum()}")
print(f"Anomalies total_load conservées         : {df_clean['is_total_load_outlier'].sum()}")

print("-" * 80)
print(f"Variables explicatives scaling base     : {feature_cols}")

print("-" * 80)
print(f"Train                                   : {train_df.shape} | {train_df.index.min()} → {train_df.index.max()}")
print(f"Validation                              : {val_df.shape} | {val_df.index.min()} → {val_df.index.max()}")
print(f"Test                                    : {test_df.shape} | {test_df.index.min()} → {test_df.index.max()}")

print("-" * 80)
print("Validation stricte :")
for key, value in validation_report.items():
    print(f"- {key}: {value}")

print("-" * 80)
print("Rapport JSON :")
print(report_path)

print("=" * 80)
print("Prétraitement terminé avec succès.")

RÉSUMÉ FINAL DU PREPROCESSING
Dimensions brutes CSV                  : (52416, 9)
Dimensions chargées avec target/total   : (52416, 10)
Dimensions après resampling horaire     : (8736, 10)
Dimensions finales propres              : (8736, 15)
--------------------------------------------------------------------------------
Période brute                           : 2017-01-01 00:00:00 → 2017-12-30 23:50:00
Période horaire                         : 2017-01-01 00:00:00 → 2017-12-30 23:00:00
Fréquence brute                         : 10min
Fréquence finale                        : h
--------------------------------------------------------------------------------
Valeurs manquantes avant traitement     : 0
Valeurs manquantes finales              : 0
Heures manquantes                       : 0
Heures avec nb mesures 10min incorrect : 0
Doublons temporels initiaux             : 0
Doublons temporels finaux               : 0
-------------------------------------------------------------------------

## 13. Rapport global détaillé du notebook

In [13]:
global_report = f"""
# Rapport global détaillé — Notebook 03 Prétraitement

## 1. Objectif général du notebook

Ce notebook correspond à la phase de prétraitement du pipeline KDD du projet **Smart City Energy Forecasting — Tetouan**. Il transforme le fichier brut `Tetuan City power consumption.csv` en datasets horaires propres, validés et prêts pour le notebook `04_feature_engineering.ipynb`.

Le prétraitement respecte la logique du projet : conserver le dataset brut dans `data/raw`, produire des fichiers propres dans `data/processed`, sauvegarder les scalers dans `models`, et générer un rapport de contrôle dans `results/preprocessing`.

## 2. Cellule 1 — Importation des bibliothèques et des fonctions `src`

Cette cellule importe les bibliothèques nécessaires au notebook :

- `pandas` pour les DataFrames et les séries temporelles ;
- `numpy` pour les contrôles numériques ;
- `Path` pour gérer les chemins de manière portable ;
- `json` pour sauvegarder le rapport de preprocessing ;
- `display` et `Markdown` pour afficher les tableaux et le rapport final dans le notebook.

Elle ajoute ensuite la racine du projet à `sys.path` afin de pouvoir importer les fonctions du dossier `src/`.

Les fonctions importées depuis `src.data_loader` centralisent le chargement du dataset, le renommage des colonnes, la conversion temporelle, la conversion numérique, la création de `target` et `total_load`, et l'audit qualité.

Les fonctions importées depuis `src.preprocessing` centralisent les opérations de preprocessing : contrôle des 6 mesures par heure, resampling horaire, recherche des heures manquantes, gestion des valeurs manquantes, contrôle physique, détection des anomalies, annotation des pics de charge, split chronologique, scaling sans fuite de données et sauvegarde des sorties.

Cette organisation rend le notebook plus professionnel : le notebook décrit et exécute le pipeline, tandis que le code réutilisable est placé dans `src/`.

## 3. Cellule 2 — Définition des chemins du projet

Cette cellule définit les chemins principaux du projet :

- fichier brut : `{RAW_PATH}` ;
- dossier des datasets transformés : `{PROCESSED_DIR}` ;
- dossier des scalers et modèles : `{MODELS_DIR}` ;
- dossier du rapport de preprocessing : `{RESULTS_DIR}`.

Le code suppose une structure GitHub professionnelle où le notebook se trouve dans `notebooks/` et les données dans `data/raw/`. Une alternative est prévue si le CSV est placé directement à la racine du projet.

Les dossiers de sortie sont créés automatiquement avec `mkdir(parents=True, exist_ok=True)`, ce qui évite les erreurs si les dossiers n'existent pas encore.

## 4. Cellule 3 — Chargement standardisé du dataset Tetouan

Le fichier brut contient `{raw_shape[0]}` observations et `{raw_shape[1]}` colonnes. La fonction `load_tetouan_data()` réalise automatiquement les étapes suivantes :

1. lecture du CSV ;
2. nettoyage des noms de colonnes, notamment les doubles espaces dans les noms des zones ;
3. renommage technique des variables ;
4. conversion de la colonne temporelle en `datetime` ;
5. conversion des variables météo et consommation en numérique ;
6. tri chronologique ;
7. suppression contrôlée des doublons temporels si nécessaire ;
8. création de `target = zone1_power` ;
9. création de `total_load = zone1_power + zone2_power + zone3_power` ;
10. indexation temporelle.

Résultat obtenu : le dataset chargé contient `{df.shape[0]}` lignes et `{df.shape[1]}` colonnes, avec une fréquence brute inférée de `{pd.infer_freq(df.index)}`.

L'audit qualité confirme que les données sont exploitables pour la suite du pipeline.

## 5. Cellule 4 — Contrôle de cohérence de `target` et `total_load`

Cette cellule vérifie deux règles essentielles du projet :

- `target` doit être exactement égale à `zone1_power` ;
- `total_load` doit être égale à la somme des trois zones.

Résultats :

- `target = zone1_power` : `{target_check}` ;
- `total_load = zone1_power + zone2_power + zone3_power` : `{total_load_check}`.

Ce contrôle est important parce que `target` est la variable cible principale du projet. Une erreur dans cette colonne fausserait le feature engineering, la modélisation et l'évaluation.

## 6. Cellule 5 — Rééchantillonnage horaire et contrôle temporel

Le dataset brut est mesuré toutes les 10 minutes. Pour construire une série horaire fiable, il ne suffit pas de faire une moyenne par heure : il faut vérifier que chaque heure contient exactement 6 mesures de 10 minutes.

La fonction `check_10min_measurements_per_hour()` vérifie cette condition. Le résultat obtenu est :

- nombre attendu de mesures par heure : `6` ;
- heures contrôlées : `{len(hourly_counts)}` ;
- heures avec un nombre incorrect de mesures : `{len(incomplete_hours)}`.

Ensuite, `resample_hourly()` calcule la moyenne horaire des variables numériques. Le dataset horaire obtenu contient `{df_hourly.shape[0]}` lignes et `{df_hourly.shape[1]}` colonnes.

La fonction `find_missing_hours()` vérifie que la grille horaire est complète. Nombre d'heures manquantes détectées : `{len(missing_hours)}`.

Ce choix est aligné avec la stratégie du projet : utiliser la granularité 10 minutes pour l'audit et l'EDA fine, puis utiliser la granularité horaire pour la modélisation principale.

## 7. Cellule 6 — Gestion sécurisée des valeurs manquantes

La fonction `handle_missing_values()` applique une stratégie prudente :

1. interpolation temporelle limitée pour les petits trous ;
2. forward fill limité ;
3. suppression contrôlée des lignes restantes si des valeurs manquantes persistent.

Le notebook évite volontairement le backfill global, car il peut utiliser de l'information future pour reconstruire le passé.

Résultats :

- valeurs manquantes avant traitement : `{missing_before}` ;
- valeurs manquantes après traitement : `{missing_after}` ;
- dimensions après nettoyage : `{df_clean.shape}`.

Comme le dataset Tetouan est déjà propre, cette étape sert surtout à rendre le pipeline robuste.

## 8. Cellule 7 — Contrôle des plages physiques et détection exploratoire des anomalies

La fonction `check_physical_ranges()` vérifie les incohérences impossibles ou suspectes :

- consommation négative ;
- humidité hors intervalle `[0, 100]` ;
- température hors plage réaliste ;
- vitesse du vent négative.

Somme des incohérences physiques détectées : `{int(range_checks.sum())}`.

La fonction `summarize_outliers()` applique ensuite un Rolling Z-Score basé sur le passé. Cette méthode compare chaque valeur à son contexte récent au lieu de la comparer à toute la distribution globale. C'est plus adapté aux séries énergétiques, car la consommation dépend fortement de l'heure, du jour et de la saison.

Les anomalies détectées ne sont pas automatiquement supprimées. Dans un projet énergétique, un pic peut représenter un vrai événement métier et non une erreur technique.

## 9. Cellule 8 — Annotation des pics de charge et validation finale stricte

La fonction `annotate_load_outliers()` ajoute les colonnes suivantes :

- `target_rolling_zscore` ;
- `total_load_rolling_zscore` ;
- `is_target_outlier` ;
- `is_total_load_outlier` ;
- `is_load_outlier`.

Les Rolling Z-Scores initiaux impossibles à calculer au début de la série sont remplacés par `0`, ce qui signifie qu'aucun écart local n'est détecté au début du dataset.

Résultats :

- anomalies sur `target` : `{int(df_clean['is_target_outlier'].sum())}` ;
- anomalies sur `total_load` : `{int(df_clean['is_total_load_outlier'].sum())}` ;
- anomalies de charge globales : `{int(df_clean['is_load_outlier'].sum())}`.

La fonction `validate_clean_hourly_dataset()` bloque le notebook si le dataset contient encore des valeurs manquantes, des doublons temporels ou une fréquence non horaire.

Validation finale :

- valeurs manquantes finales : `{validation_report['missing_total']}` ;
- doublons finaux : `{validation_report['duplicated_total']}` ;
- fréquence finale : `{validation_report['inferred_frequency']}`.

## 10. Cellule 9 — Split chronologique train / validation / test

La fonction `temporal_train_val_test_split()` découpe le dataset selon l'ordre temporel :

- train : passé ;
- validation : période intermédiaire ;
- test : futur.

Aucun shuffle n'est utilisé, car mélanger le passé et le futur créerait une fuite d'information.

Résultats :

- train : `{train_df.shape}` de `{train_df.index.min()}` à `{train_df.index.max()}` ;
- validation : `{val_df.shape}` de `{val_df.index.min()}` à `{val_df.index.max()}` ;
- test : `{test_df.shape}` de `{test_df.index.min()}` à `{test_df.index.max()}`.

Ce découpage prépare correctement la modélisation temporelle.

## 11. Cellule 10 — Normalisation de base sans data leakage

La normalisation est faite avec `scale_train_val_test()`.

Règle appliquée :

- le scaler des features est ajusté uniquement sur `train_df` ;
- le scaler de la cible est ajusté uniquement sur `train_df` ;
- validation et test sont seulement transformés.

Les variables explicatives utilisées dans ce notebook sont uniquement les variables météo de base :

`{feature_cols}`

Les colonnes de consommation instantanée (`target`, `zone1_power`, `zone2_power`, `zone3_power`, `total_load`) sont conservées dans les fichiers propres, mais elles ne sont pas utilisées directement comme variables explicatives. Elles seront utilisées plus tard uniquement sous forme de lags ou de rolling features basées sur le passé.

Cette règle évite le data leakage.

## 12. Cellule 11 — Sauvegarde des datasets, fichiers normalisés, scalers et rapport JSON

La fonction `save_preprocessing_outputs()` sauvegarde les fichiers suivants dans `data/processed` :

- dataset horaire complet nettoyé ;
- train propre ;
- validation propre ;
- test propre ;
- fichiers normalisés de base `X` et `y`.

La fonction `save_scalers()` sauvegarde les scalers dans `models`.

Le rapport JSON est sauvegardé ici :

`{report_path}`

Ce rapport permet de tracer automatiquement les dimensions, la période couverte, les fréquences, les valeurs manquantes, les doublons, les anomalies, les features utilisées et les fichiers sauvegardés.

## 13. Cellule 12 — Résumé final du preprocessing

Cette cellule affiche un résumé compact du pipeline : dimensions, période, fréquence, valeurs manquantes, doublons, anomalies, features de base, splits et rapport sauvegardé.

Elle confirme que le notebook a produit un dataset propre, horaire, sans valeurs manquantes, sans doublons et prêt pour le feature engineering.

## 14. Règles importantes pour le notebook `04_feature_engineering.ipynb`

Les fichiers propres conservent les colonnes de consommation instantanée. Cela est volontaire, car ces colonnes sont nécessaires pour construire des variables retardées.

Règle autorisée :

- `target_lag_1h` ;
- `target_lag_24h` ;
- `target_lag_168h` ;
- rolling mean/std/min/max calculés avec `shift(1)`.

Règle interdite :

- utiliser `target`, `zone1_power`, `zone2_power`, `zone3_power` ou `total_load` au même timestamp comme variables explicatives directes ;
- utiliser les z-scores de charge et les indicateurs d'outliers de charge comme features prédictives directes si ces informations ne sont pas disponibles au moment de la prédiction.

## 15. Verdict final

Le notebook simplifié est aligné avec le projet et avec l'architecture GitHub attendue. Il conserve les garanties scientifiques de l'ancien notebook, mais il supprime les définitions longues et répétitives en utilisant les fonctions du dossier `src/`.

Le preprocessing est validé :

- dataset brut chargé correctement ;
- fréquence 10 minutes contrôlée ;
- resampling horaire validé ;
- aucune heure manquante ;
- aucune valeur manquante finale ;
- aucune incohérence physique détectée ;
- anomalies documentées et conservées ;
- split chronologique correct ;
- normalisation sans data leakage ;
- sorties sauvegardées pour le notebook suivant.
"""

display(Markdown(global_report))

global_report_path = RESULTS_DIR / "preprocessing_global_report.md"
global_report_path.write_text(global_report, encoding="utf-8")

print(f"Rapport global détaillé sauvegardé : {global_report_path}")


# Rapport global détaillé — Notebook 03 Prétraitement

## 1. Objectif général du notebook

Ce notebook correspond à la phase de prétraitement du pipeline KDD du projet **Smart City Energy Forecasting — Tetouan**. Il transforme le fichier brut `Tetuan City power consumption.csv` en datasets horaires propres, validés et prêts pour le notebook `04_feature_engineering.ipynb`.

Le prétraitement respecte la logique du projet : conserver le dataset brut dans `data/raw`, produire des fichiers propres dans `data/processed`, sauvegarder les scalers dans `models`, et générer un rapport de contrôle dans `results/preprocessing`.

## 2. Cellule 1 — Importation des bibliothèques et des fonctions `src`

Cette cellule importe les bibliothèques nécessaires au notebook :

- `pandas` pour les DataFrames et les séries temporelles ;
- `numpy` pour les contrôles numériques ;
- `Path` pour gérer les chemins de manière portable ;
- `json` pour sauvegarder le rapport de preprocessing ;
- `display` et `Markdown` pour afficher les tableaux et le rapport final dans le notebook.

Elle ajoute ensuite la racine du projet à `sys.path` afin de pouvoir importer les fonctions du dossier `src/`.

Les fonctions importées depuis `src.data_loader` centralisent le chargement du dataset, le renommage des colonnes, la conversion temporelle, la conversion numérique, la création de `target` et `total_load`, et l'audit qualité.

Les fonctions importées depuis `src.preprocessing` centralisent les opérations de preprocessing : contrôle des 6 mesures par heure, resampling horaire, recherche des heures manquantes, gestion des valeurs manquantes, contrôle physique, détection des anomalies, annotation des pics de charge, split chronologique, scaling sans fuite de données et sauvegarde des sorties.

Cette organisation rend le notebook plus professionnel : le notebook décrit et exécute le pipeline, tandis que le code réutilisable est placé dans `src/`.

## 3. Cellule 2 — Définition des chemins du projet

Cette cellule définit les chemins principaux du projet :

- fichier brut : `C:\Users\Badr\UB\0_Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\data\raw\Tetuan City power consumption.csv` ;
- dossier des datasets transformés : `C:\Users\Badr\UB\0_Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\data\processed` ;
- dossier des scalers et modèles : `C:\Users\Badr\UB\0_Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\models` ;
- dossier du rapport de preprocessing : `C:\Users\Badr\UB\0_Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\results\preprocessing`.

Le code suppose une structure GitHub professionnelle où le notebook se trouve dans `notebooks/` et les données dans `data/raw/`. Une alternative est prévue si le CSV est placé directement à la racine du projet.

Les dossiers de sortie sont créés automatiquement avec `mkdir(parents=True, exist_ok=True)`, ce qui évite les erreurs si les dossiers n'existent pas encore.

## 4. Cellule 3 — Chargement standardisé du dataset Tetouan

Le fichier brut contient `52416` observations et `9` colonnes. La fonction `load_tetouan_data()` réalise automatiquement les étapes suivantes :

1. lecture du CSV ;
2. nettoyage des noms de colonnes, notamment les doubles espaces dans les noms des zones ;
3. renommage technique des variables ;
4. conversion de la colonne temporelle en `datetime` ;
5. conversion des variables météo et consommation en numérique ;
6. tri chronologique ;
7. suppression contrôlée des doublons temporels si nécessaire ;
8. création de `target = zone1_power` ;
9. création de `total_load = zone1_power + zone2_power + zone3_power` ;
10. indexation temporelle.

Résultat obtenu : le dataset chargé contient `52416` lignes et `10` colonnes, avec une fréquence brute inférée de `10min`.

L'audit qualité confirme que les données sont exploitables pour la suite du pipeline.

## 5. Cellule 4 — Contrôle de cohérence de `target` et `total_load`

Cette cellule vérifie deux règles essentielles du projet :

- `target` doit être exactement égale à `zone1_power` ;
- `total_load` doit être égale à la somme des trois zones.

Résultats :

- `target = zone1_power` : `True` ;
- `total_load = zone1_power + zone2_power + zone3_power` : `True`.

Ce contrôle est important parce que `target` est la variable cible principale du projet. Une erreur dans cette colonne fausserait le feature engineering, la modélisation et l'évaluation.

## 6. Cellule 5 — Rééchantillonnage horaire et contrôle temporel

Le dataset brut est mesuré toutes les 10 minutes. Pour construire une série horaire fiable, il ne suffit pas de faire une moyenne par heure : il faut vérifier que chaque heure contient exactement 6 mesures de 10 minutes.

La fonction `check_10min_measurements_per_hour()` vérifie cette condition. Le résultat obtenu est :

- nombre attendu de mesures par heure : `6` ;
- heures contrôlées : `8736` ;
- heures avec un nombre incorrect de mesures : `0`.

Ensuite, `resample_hourly()` calcule la moyenne horaire des variables numériques. Le dataset horaire obtenu contient `8736` lignes et `10` colonnes.

La fonction `find_missing_hours()` vérifie que la grille horaire est complète. Nombre d'heures manquantes détectées : `0`.

Ce choix est aligné avec la stratégie du projet : utiliser la granularité 10 minutes pour l'audit et l'EDA fine, puis utiliser la granularité horaire pour la modélisation principale.

## 7. Cellule 6 — Gestion sécurisée des valeurs manquantes

La fonction `handle_missing_values()` applique une stratégie prudente :

1. interpolation temporelle limitée pour les petits trous ;
2. forward fill limité ;
3. suppression contrôlée des lignes restantes si des valeurs manquantes persistent.

Le notebook évite volontairement le backfill global, car il peut utiliser de l'information future pour reconstruire le passé.

Résultats :

- valeurs manquantes avant traitement : `0` ;
- valeurs manquantes après traitement : `0` ;
- dimensions après nettoyage : `(8736, 15)`.

Comme le dataset Tetouan est déjà propre, cette étape sert surtout à rendre le pipeline robuste.

## 8. Cellule 7 — Contrôle des plages physiques et détection exploratoire des anomalies

La fonction `check_physical_ranges()` vérifie les incohérences impossibles ou suspectes :

- consommation négative ;
- humidité hors intervalle `[0, 100]` ;
- température hors plage réaliste ;
- vitesse du vent négative.

Somme des incohérences physiques détectées : `0`.

La fonction `summarize_outliers()` applique ensuite un Rolling Z-Score basé sur le passé. Cette méthode compare chaque valeur à son contexte récent au lieu de la comparer à toute la distribution globale. C'est plus adapté aux séries énergétiques, car la consommation dépend fortement de l'heure, du jour et de la saison.

Les anomalies détectées ne sont pas automatiquement supprimées. Dans un projet énergétique, un pic peut représenter un vrai événement métier et non une erreur technique.

## 9. Cellule 8 — Annotation des pics de charge et validation finale stricte

La fonction `annotate_load_outliers()` ajoute les colonnes suivantes :

- `target_rolling_zscore` ;
- `total_load_rolling_zscore` ;
- `is_target_outlier` ;
- `is_total_load_outlier` ;
- `is_load_outlier`.

Les Rolling Z-Scores initiaux impossibles à calculer au début de la série sont remplacés par `0`, ce qui signifie qu'aucun écart local n'est détecté au début du dataset.

Résultats :

- anomalies sur `target` : `0` ;
- anomalies sur `total_load` : `0` ;
- anomalies de charge globales : `0`.

La fonction `validate_clean_hourly_dataset()` bloque le notebook si le dataset contient encore des valeurs manquantes, des doublons temporels ou une fréquence non horaire.

Validation finale :

- valeurs manquantes finales : `0` ;
- doublons finaux : `0` ;
- fréquence finale : `h`.

## 10. Cellule 9 — Split chronologique train / validation / test

La fonction `temporal_train_val_test_split()` découpe le dataset selon l'ordre temporel :

- train : passé ;
- validation : période intermédiaire ;
- test : futur.

Aucun shuffle n'est utilisé, car mélanger le passé et le futur créerait une fuite d'information.

Résultats :

- train : `(6115, 15)` de `2017-01-01 00:00:00` à `2017-09-12 18:00:00` ;
- validation : `(1310, 15)` de `2017-09-12 19:00:00` à `2017-11-06 08:00:00` ;
- test : `(1311, 15)` de `2017-11-06 09:00:00` à `2017-12-30 23:00:00`.

Ce découpage prépare correctement la modélisation temporelle.

## 11. Cellule 10 — Normalisation de base sans data leakage

La normalisation est faite avec `scale_train_val_test()`.

Règle appliquée :

- le scaler des features est ajusté uniquement sur `train_df` ;
- le scaler de la cible est ajusté uniquement sur `train_df` ;
- validation et test sont seulement transformés.

Les variables explicatives utilisées dans ce notebook sont uniquement les variables météo de base :

`['temperature', 'humidity', 'wind_speed', 'general_diffuse_flows', 'diffuse_flows']`

Les colonnes de consommation instantanée (`target`, `zone1_power`, `zone2_power`, `zone3_power`, `total_load`) sont conservées dans les fichiers propres, mais elles ne sont pas utilisées directement comme variables explicatives. Elles seront utilisées plus tard uniquement sous forme de lags ou de rolling features basées sur le passé.

Cette règle évite le data leakage.

## 12. Cellule 11 — Sauvegarde des datasets, fichiers normalisés, scalers et rapport JSON

La fonction `save_preprocessing_outputs()` sauvegarde les fichiers suivants dans `data/processed` :

- dataset horaire complet nettoyé ;
- train propre ;
- validation propre ;
- test propre ;
- fichiers normalisés de base `X` et `y`.

La fonction `save_scalers()` sauvegarde les scalers dans `models`.

Le rapport JSON est sauvegardé ici :

`C:\Users\Badr\UB\0_Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\results\preprocessing\preprocessing_report.json`

Ce rapport permet de tracer automatiquement les dimensions, la période couverte, les fréquences, les valeurs manquantes, les doublons, les anomalies, les features utilisées et les fichiers sauvegardés.

## 13. Cellule 12 — Résumé final du preprocessing

Cette cellule affiche un résumé compact du pipeline : dimensions, période, fréquence, valeurs manquantes, doublons, anomalies, features de base, splits et rapport sauvegardé.

Elle confirme que le notebook a produit un dataset propre, horaire, sans valeurs manquantes, sans doublons et prêt pour le feature engineering.

## 14. Règles importantes pour le notebook `04_feature_engineering.ipynb`

Les fichiers propres conservent les colonnes de consommation instantanée. Cela est volontaire, car ces colonnes sont nécessaires pour construire des variables retardées.

Règle autorisée :

- `target_lag_1h` ;
- `target_lag_24h` ;
- `target_lag_168h` ;
- rolling mean/std/min/max calculés avec `shift(1)`.

Règle interdite :

- utiliser `target`, `zone1_power`, `zone2_power`, `zone3_power` ou `total_load` au même timestamp comme variables explicatives directes ;
- utiliser les z-scores de charge et les indicateurs d'outliers de charge comme features prédictives directes si ces informations ne sont pas disponibles au moment de la prédiction.

## 15. Verdict final

Le notebook simplifié est aligné avec le projet et avec l'architecture GitHub attendue. Il conserve les garanties scientifiques de l'ancien notebook, mais il supprime les définitions longues et répétitives en utilisant les fonctions du dossier `src/`.

Le preprocessing est validé :

- dataset brut chargé correctement ;
- fréquence 10 minutes contrôlée ;
- resampling horaire validé ;
- aucune heure manquante ;
- aucune valeur manquante finale ;
- aucune incohérence physique détectée ;
- anomalies documentées et conservées ;
- split chronologique correct ;
- normalisation sans data leakage ;
- sorties sauvegardées pour le notebook suivant.


Rapport global détaillé sauvegardé : C:\Users\Badr\UB\0_Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\results\preprocessing\preprocessing_global_report.md
